# 04 — Evaluation handoff

Use this after `03_training.ipynb`. Today, the repository can reproduce the A/B **training/validation comparison** by running the comparison cell in notebook 03 after both full runs finish. This notebook only checks those prerequisites; it does not calculate model scores. The fixed ten-stage recognition runner described in the model plan is not implemented, so do not report ten-stage results from this notebook or from training curves.

## Choose the result you want to reproduce

1. **A/B training-validation comparison (implemented):** open `03_training.ipynb`, select the same profile and full checkpoint root used for training, and set `RUN_COMPARISON = True` after both A and B finish. The existing `training.compare_ab` script writes `comparison.md`, `comparison-summary.json`, validation-curve plots, and per-run plots under `<checkpoint-root>/ab_comparison/`.

2. **Fixed ten-stage recognition evaluation (not implemented):** the model plan describes the intended protocol, but the repository currently has no prediction/scoring runner or report command for it. Do not treat A/B training curves as recognition scores. The planned protocol is documented in [the Guesser model plan, Section 10](../../../../docs/guesser/guesser-model-plan.md).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from guesser_evaluation.notebook_tools import module_source_path

# Derive roots from the checked-out package so the notebook remains clone-portable.
PACKAGE_SOURCE = module_source_path("notebook_tools.py")
REPOSITORY_ROOT = PACKAGE_SOURCE.parents[5]
MODEL_PROJECT = REPOSITORY_ROOT / "models" / "guesser"

PROFILE = "production"  # Use the same prepared profile used for A/B training.
CHECKPOINT_ROOT = MODEL_PROJECT / "checkpoints" / f"notebook-{PROFILE}-full"
PREPARED_DIR = MODEL_PROJECT / "data" / ".local" / "data" / PROFILE / "prepared"
if PROFILE not in {"small", "medium", "production"}:
    raise ValueError("Select small, medium, or production.")
print("Profile:", PROFILE)
print("Prepared validation data:", PREPARED_DIR)
print("A/B checkpoint root:", CHECKPOINT_ROOT)

## Check A/B comparison inputs

Set `PROFILE` and `CHECKPOINT_ROOT` below to the same values as `DATA_PROFILE` and `CHECKPOINT_ROOT` in notebook 03. This read-only check verifies the validation manifest and both training-arm checkpoint handoffs. It does not read test data, load the checkpoints, or run inference.

In [ ]:
import json

validation_manifest = PREPARED_DIR / "manifests" / "validation.parquet"
selection_paths = {
    "A_complete_only": CHECKPOINT_ROOT / "A_complete_only" / "checkpoint-selection.json",
    "B_partial": CHECKPOINT_ROOT / "B_partial" / "checkpoint-selection.json",
}
print("Validation manifest present:", validation_manifest.is_file())
if not validation_manifest.is_file():
    print("Validation preparation is incomplete; run 02_dataset_audit after preparing the profile.")

checkpoint_handoffs = {}
for run_name, selection_path in selection_paths.items():
    if not selection_path.is_file():
        checkpoint_handoffs[run_name] = {"ready": False, "reason": "training handoff not found"}
        continue
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    checkpoint_value = selection.get("best_model_path")
    checkpoint_path = Path(checkpoint_value) if checkpoint_value else None
    if checkpoint_path is not None and not checkpoint_path.is_absolute():
        checkpoint_path = MODEL_PROJECT / checkpoint_path
    checkpoint_handoffs[run_name] = {
        "ready": checkpoint_path is not None and checkpoint_path.is_file(),
        "selection_file": str(selection_path),
        "best_model_path": str(checkpoint_path) if checkpoint_path else None,
        "checkpoint_exists": checkpoint_path.is_file() if checkpoint_path else False,
    }
for run_name, handoff in checkpoint_handoffs.items():
    print(run_name, handoff)

ready = validation_manifest.is_file() and all(
    item["ready"] for item in checkpoint_handoffs.values()
)
if ready:
    print(
        "A/B inputs are present. Next: set RUN_COMPARISON = True in 03_training.ipynb and run its comparison cell."
    )
else:
    print(
        "A/B comparison is not ready. In 03_training.ipynb, use this same profile/root and finish both full training runs; then rerun this check."
    )

## What to do next

- If either A/B handoff is missing, return to notebook 03 and finish that arm using the same profile, settings, and checkpoint root.
- If both handoffs are present, return to notebook 03 and run its A/B comparison cell. That is the reproducible model result currently supported by repository code.
- For ten-stage recognition scores, wait until a prediction/scoring runner exists. Then run it on validation first; use validation for checkpoint selection and calibration, and run held-out test only after the model and protocol are frozen. Until then, record ten-stage evaluation as **not run**.